# AlphaTrain Path B — SMOKE (10 epochs per run)

Short-form version of `train_path_b_colab.ipynb`. **10 epochs × 3 runs ≈ 9h total** on an H100, fits comfortably under Colab's 24h kill timer.

## Purpose: trend check, not absolute peak

pillar2z trained 40 epochs and val_loss was still falling. At 10 epochs we will NOT hit its absolute best. The gate becomes: **does Run B's epoch-10 val_loss trend match pillar2z's epoch-10 val_loss?**

Pull pillar2z's epoch-10 val from its training log on Drive — that's the comparable target.

## Ablation matrix (10 epochs each)

| run | --oracle-lambda | --color-augment | purpose |
|---|---|---|---|
| **B** | 0.0  | yes | Isolate gain from color symmetry (sanity vs pillar2z) |
| **C** | 0.05 | yes | Conservative oracle pressure |
| **D** | 0.10 | yes | Aggressive oracle pressure |

## What to watch in C/D

Oracle anchors get ~72× oversample/epoch → ~720× across 10 epochs. Plenty of exposure. By epoch 5-7, `top1≥.15` and `KL_weighted` should move if oracle is doing real work. If they don't move, λ is too small. If V12 val_loss diverges from B's trajectory, λ is too large.

If a smoke run looks promising, extend with `--resume <epoch_10.pt>` in the full 40-epoch notebook.

## Upload to Drive (`MyDrive/alphatrain/`)

1. `colorlines_path_b.tar.gz` — code + oracle tensor (~1.1 MB)
2. `v12_pillar2z.pt.gz` — V12 tensor, already there from pillar2z run (~2-3 GB)
3. `pillar2y2_epoch_40.pt` — warm-start, already there (~143 MB)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_path_b.tar.gz /content/
!cd /content && tar xzf colorlines_path_b.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy(f'{DRIVE}/pillar2y2_epoch_40.pt',
            '/content/alphatrain/data/model.pt')
print(f'Warm-start: {os.path.getsize("/content/alphatrain/data/model.pt")/1e6:.0f} MB')

t0 = time.time()
!gzip -dc {DRIVE}/v12_pillar2z.pt.gz > /content/alphatrain/data/selfplay.pt
print(f'V12 tensor: {os.path.getsize("/content/alphatrain/data/selfplay.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

print(f'Oracle tensor: {os.path.getsize("/content/alphatrain/data/phase1_oracle_path_b.pt")/1e6:.1f} MB')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

%cd /content
!python -m pytest alphatrain/tests/test_train_path_b.py -v --tb=short 2>&1 | tail -20

## Run B (smoke) — V12 + color aug, λ=0

~3h on H100. λ=0 → oracle tensor not loaded. Sanity check vs pillar2z's *epoch-10* val_loss (not best).

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/selfplay.pt \
    --amp --compile \
    --resume alphatrain/data/model.pt --warm-start \
    --color-augment \
    --epochs 10 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
    --oracle-lambda 0.0 \
    --copy-to /content/drive/MyDrive/alphatrain/path_b_B_smoke_best.pt \
    --save-dir /content/checkpoints/path_b_B_smoke

In [ ]:
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
RUN = 'path_b_B_smoke'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst = f'{DRIVE}/{RUN}_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/{RUN}_{f}')
        print(f'Saved {DRIVE}/{RUN}_{f}')

## Run C (smoke) — λ=0.05

~3h. Tracks oracle val metrics every epoch (KL_all, KL_weighted, top1 by margin bucket).

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/selfplay.pt \
    --oracle-tensor alphatrain/data/phase1_oracle_path_b.pt \
    --oracle-lambda 0.05 --oracle-beta 10.0 \
    --oracle-noise-floor 0.05 --oracle-scale 0.20 \
    --oracle-batch-size 4096 \
    --amp --compile \
    --resume alphatrain/data/model.pt --warm-start \
    --color-augment \
    --epochs 10 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
    --copy-to /content/drive/MyDrive/alphatrain/path_b_C_smoke_best.pt \
    --save-dir /content/checkpoints/path_b_C_smoke

In [ ]:
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
RUN = 'path_b_C_smoke'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst = f'{DRIVE}/{RUN}_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/{RUN}_{f}')
        print(f'Saved {DRIVE}/{RUN}_{f}')

## Run D (smoke) — λ=0.10

~3h. Same as C with 2× oracle pressure.

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/selfplay.pt \
    --oracle-tensor alphatrain/data/phase1_oracle_path_b.pt \
    --oracle-lambda 0.10 --oracle-beta 10.0 \
    --oracle-noise-floor 0.05 --oracle-scale 0.20 \
    --oracle-batch-size 4096 \
    --amp --compile \
    --resume alphatrain/data/model.pt --warm-start \
    --color-augment \
    --epochs 10 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
    --copy-to /content/drive/MyDrive/alphatrain/path_b_D_smoke_best.pt \
    --save-dir /content/checkpoints/path_b_D_smoke

In [ ]:
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
RUN = 'path_b_D_smoke'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst = f'{DRIVE}/{RUN}_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/{RUN}_{f}')
        print(f'Saved {DRIVE}/{RUN}_{f}')